# Fit the Models on Train and Measure Them on Train

## Natural-language request

> Create a notebook which applies all the models discussed (logistic regression, random forest, XGBoost, CatBoost), apply and compute on the training set.

This notebook fits four models on the **training set**, using the inputs prepared and leakage-checked in notebook 10, and measures each one **on the same training set**. A **flat-rule reference** sits alongside them: each hotel's train cancellation rate, given to every booking at that hotel. That is the rate part of the client's flat rule (`docs/EVALUATION_CONTRACT.md`).

## Read every number here as a training-set fit, not as performance

- **These are in-sample numbers.** Each model is measured on the bookings it learnt from. Flexible models such as the random forest and XGBoost can partly memorise train, so their train scores flatter them. **No model is chosen here.** Choosing happens on val (project rule 2), and nothing here is a performance or savings claim (project rule 8).
- **No euro cost is computed.** The contract's cost needs an oversell buffer, and the buffer must be chosen on val under the walk ceiling (project rules 5 and 6). This notebook stops at the probabilities and the expected cancellations per hotel-night.
- **Four inputs are provisional** (`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `company`): used although their timing is not verified (notebook 06). Every result here carries that caveat.
- **Val and test are not opened.**

## What is measured

| Measure | What it says | Better |
|---|---|---|
| **Log loss** | How far the probabilities are from what happened, with confident mistakes punished hard | Lower |
| **Brier score** | Mean squared gap between probability and outcome (0 or 1) | Lower |
| **ROC AUC** | How well the model ranks cancellations above stays; 0.5 is guessing | Higher |
| **Mean predicted vs actual rate** | Whether the probabilities add up to the right overall level | Close together |
| **Expected vs actual cancellations per hotel-night** | The quantity the oversell decision is built on: `contract.nightly_table` adds each night's probabilities | Smaller error |

The last measure matters most for this project. The contract adds up probabilities night by night, so a model that ranks well but whose probabilities are too high or too low would oversell the wrong number of rooms.

## Setup

Loads the prepared train table and the fitted preprocessor's column names from notebook 10, and the train labels and scoring columns from notebook 08. Only train is opened.

In [1]:
from pathlib import Path
import json
import sys
import time

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from xgboost import XGBClassifier

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

split_dir = repo_root / "data" / "splits"
prep_dir = repo_root / "data" / "preprocessed"
if not (prep_dir / "train_X.csv").exists():
    raise FileNotFoundError("data/preprocessed/ is missing. Run notebooks/10_preprocessing_leakage_checked.ipynb first.")

manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))
prep_info = json.loads((prep_dir / "feature_names.json").read_text(encoding="utf-8"))
label = manifest["label"]

X_train = pd.read_csv(prep_dir / "train_X.csv", index_col="row")
train = pd.read_csv(split_dir / "train.csv", index_col="row", parse_dates=["arrival_date"],
                    dtype={"agent": str, "company": str})
y_train = train.loc[X_train.index, label]

print(f"X_train {X_train.shape}, y_train {y_train.shape}, train cancellation rate {y_train.mean():.1%}")

X_train (49157, 143), y_train (49157,), train cancellation rate 35.8%


## Leakage guards before fitting

Notebook 10 checked the preprocessing in detail. These guards confirm the tables reaching the models are still those:

1. The prepared columns are exactly the ones notebook 10 saved, so no label, date, price or identifier column is among them.
2. Every prepared row has its label, matched by booking row number, not by position.
3. Every row is a train booking (arrival before 2016-07-01).

In [2]:
assert list(X_train.columns) == prep_info["output_columns"], "prepared columns differ from notebook 10"
assert label not in X_train.columns and not set(manifest["scoring_only"]) & set(X_train.columns)
assert X_train.index.equals(y_train.index) and X_train.index.isin(train.index).all()
assert train.loc[X_train.index, "arrival_date"].max() < pd.Timestamp(contract.TRAIN_END)
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
print(f"Guards pass: {X_train.shape[1]} prepared columns from notebook 10, {len(X_train):,} train bookings, labels matched by row.")

Guards pass: 143 prepared columns from notebook 10, 49,157 train bookings, labels matched by row.


## The models and their settings

All four use **fixed, modest settings chosen before seeing any result**. They are not tuned: tuning is a choice and belongs on val (project rule 2). `random_state` is fixed so every run gives the same result.

| Model | Settings | Why these |
|---|---|---|
| **Flat-rule reference** | Each hotel's train cancellation rate | The rate part of the client's flat rule; every model should beat it. |
| **Logistic regression** | Default regularisation (C = 1), up to 2,000 iterations | A simple, readable model; the inputs are already scaled (notebook 10). |
| **Random forest** | 300 trees, at least 20 bookings per leaf | The leaf minimum stops trees from memorising single bookings. |
| **XGBoost** | 300 rounds, depth 4, learning rate 0.05, 80% of rows and columns per round | Shallow trees and a slow learning rate are a cautious starting point. |
| **CatBoost** | 300 rounds, depth 4, learning rate 0.05 | Same budget as XGBoost, so the two can be compared fairly. |

**No class weights or resampling.** The cancellation rate is 35.8%, and rebalancing would push the probabilities away from the real rate. The contract adds the probabilities up, so they must stay on the real scale.

All four models use the **same prepared table** from notebook 10. CatBoost can also take raw categories and encode them itself using the label. That would be a new, label-based preprocessing step needing its own leakage review, so it is not used here.

In [3]:
SEED = 0
models = {
    "logistic regression": LogisticRegression(C=1.0, max_iter=2000),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             eval_metric="logloss", random_state=SEED, n_jobs=-1),
    "CatBoost": CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05, random_seed=SEED, verbose=0,
                                   allow_writing_files=False),
}
print("Models:", ["flat-rule reference", *models])

Models: ['flat-rule reference', 'logistic regression', 'random forest', 'XGBoost', 'CatBoost']


## Fit every model on train, then predict train

Each model is fitted on train only and then gives every train booking a cancellation probability. The flat-rule reference needs no fitting: it is each hotel's share of cancelled train bookings.

In [4]:
X = X_train.to_numpy(dtype=float)
probabilities = {}
fit_seconds = {}

hotel_rate = y_train.groupby(train.loc[X_train.index, "hotel"]).mean()
probabilities["flat-rule reference"] = train.loc[X_train.index, "hotel"].map(hotel_rate).to_numpy()
fit_seconds["flat-rule reference"] = 0.0
print("Flat-rule reference rates (train):", {h: f"{r:.1%}" for h, r in hotel_rate.items()})

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X, y_train.to_numpy())
    fit_seconds[name] = time.perf_counter() - start
    probabilities[name] = model.predict_proba(X)[:, 1]
    print(f"Fitted {name} in {fit_seconds[name]:.1f} s")

for name, p in probabilities.items():
    assert len(p) == len(y_train) and np.all((p >= 0) & (p <= 1)), f"{name}: invalid probabilities"

Flat-rule reference rates (train): {'City Hotel': '41.7%', 'Resort Hotel': '25.2%'}


Fitted logistic regression in 1.4 s


Fitted random forest in 4.5 s


Fitted XGBoost in 3.7 s


Fitted CatBoost in 5.1 s


## Training-set scores

Probability scores for every booking in train. Again: **in-sample**. A big gap between a flexible model and logistic regression here may be partly memorisation, and only val can show how much.

In [5]:
scores = pd.DataFrame({
    name: {
        "log loss": log_loss(y_train, p),
        "Brier score": brier_score_loss(y_train, p),
        "ROC AUC": roc_auc_score(y_train, p),
        "mean predicted rate": p.mean(),
        "actual rate": y_train.mean(),
        "fit time (s)": fit_seconds[name],
    }
    for name, p in probabilities.items()
}).T
display(scores.style.format({"log loss": "{:.4f}", "Brier score": "{:.4f}", "ROC AUC": "{:.3f}",
                             "mean predicted rate": "{:.1%}", "actual rate": "{:.1%}", "fit time (s)": "{:.1f}"}))

,log loss,Brier score,ROC AUC,mean predicted rate,actual rate,fit time (s)
flat-rule reference,0.6381,0.2235,0.582,35.8%,35.8%,0.0
logistic regression,0.4622,0.1518,0.817,35.8%,35.8%,1.4
random forest,0.4802,0.1573,0.810,35.8%,35.8%,4.5
XGBoost,0.4576,0.1522,0.815,35.8%,35.8%,3.7
CatBoost,0.4589,0.1524,0.815,35.8%,35.8%,5.1


## Expected vs actual cancellations per hotel-night

`contract.nightly_table` adds each night's probabilities into the **expected** number of cancellations and sets it beside what actually happened. The oversell decision is built on this number, so these are the measures closest to the client's problem:

- **mean absolute error**: on an average night, how many rooms the expected count is off by;
- **mean error** (expected minus actual): above zero, the model expects too many cancellations, which would push towards overselling; below zero, too few;
- **share of nights within one room**.

Results are shown for the group and for each hotel separately (project rule 6).

In [6]:
scoring_columns = train.loc[X_train.index, contract.REQUIRED]
nightly = {name: contract.nightly_table(scoring_columns, p) for name, p in probabilities.items()}

rows = []
for name, table in nightly.items():
    error = table["expected"] - table["cancelled"]
    for hotel, part in [("group", error), *error.groupby(level="hotel")]:
        rows.append({"model": name, "hotel": hotel, "hotel-nights": len(part),
                     "mean absolute error (rooms)": part.abs().mean(),
                     "mean error (rooms)": part.mean(),
                     "nights within 1 room": (part.abs() <= 1).mean()})
nightly_scores = pd.DataFrame(rows).set_index(["hotel", "model"]).sort_index()
display(nightly_scores.style.format({"mean absolute error (rooms)": "{:.2f}", "mean error (rooms)": "{:+.2f}",
                                     "nights within 1 room": "{:.1%}"}))

## Calibration: do the probabilities mean what they say?

Train bookings are grouped by each model's probability into fixed bands (0–10%, 10–20%, …, 90–100%). For each band the table shows the **mean predicted probability**, the **share that actually cancelled**, and the number of bookings. For a well-calibrated model the first two are close in every band.

Fixed bands keep bookings with **the same probability together**. Many bookings share identical inputs and so get identical probabilities; splitting them into equal-sized groups would cut those ties apart arbitrarily and show gaps that are not real.

The last row is the **average gap**, weighted by the number of bookings in each band.

In [7]:
bands = np.linspace(0, 1, 11)
band_names = [f"{int(lo * 100)}-{int(hi * 100)}%" for lo, hi in zip(bands[:-1], bands[1:])]
calibration = {}
for name, p in probabilities.items():
    band = pd.cut(p, bands, labels=band_names, include_lowest=True)
    by_band = (pd.DataFrame({"predicted": p, "actual": y_train.to_numpy(), "band": band})
               .groupby("band", observed=False).agg(predicted=("predicted", "mean"), actual=("actual", "mean"),
                                                    bookings=("actual", "size")))
    cells = [f"{r.predicted:.0%} vs {r.actual:.0%} ({r.bookings:,})" if r.bookings else "-" for r in by_band.itertuples()]
    filled = by_band[by_band["bookings"] > 0]
    gap = ((filled["predicted"] - filled["actual"]).abs() * filled["bookings"]).sum() / filled["bookings"].sum()
    calibration[name] = cells + [f"{gap:.1%}"]
display(pd.DataFrame(calibration, index=[f"{b}: predicted vs actual (bookings)" for b in band_names] + ["average gap (weighted)"]))

,flat-rule reference,logistic regression,random forest,XGBoost,CatBoost
0-10%: predicted vs actual (bookings),-,"5% vs 6% (7,096)","8% vs 3% (2,542)","6% vs 4% (3,727)","6% vs 3% (3,040)"
10-20%: predicted vs actual (bookings),-,"14% vs 13% (8,371)","15% vs 11% (11,274)","15% vs 12% (11,893)","14% vs 11% (12,190)"
20-30%: predicted vs actual (bookings),"25% vs 25% (17,534)","28% vs 27% (8,212)","27% vs 24% (8,361)","27% vs 27% (7,326)","27% vs 26% (7,750)"
30-40%: predicted vs actual (bookings),-,"33% vs 33% (11,086)","33% vs 32% (12,557)","32% vs 33% (12,410)","32% vs 33% (12,457)"
40-50%: predicted vs actual (bookings),"42% vs 42% (31,623)","45% vs 45% (5,165)","45% vs 48% (4,014)","45% vs 47% (6,067)","45% vs 46% (5,810)"
50-60%: predicted vs actual (bookings),-,"53% vs 57% (1,998)","53% vs 52% (4,106)",53% vs 62% (990),"54% vs 64% (1,451)"
60-70%: predicted vs actual (bookings),-,66% vs 62% (551),63% vs 80% (862),64% vs 70% (813),66% vs 72% (515)
70-80%: predicted vs actual (bookings),-,73% vs 75% (715),73% vs 100% (700),73% vs 100% (67),76% vs 92% (114)
80-90%: predicted vs actual (bookings),-,85% vs 88% (219),"89% vs 100% (3,072)",81% vs 98% (122),87% vs 96% (159)
90-100%: predicted vs actual (bookings),-,"98% vs 99% (5,744)","93% vs 100% (1,669)","99% vs 100% (5,742)","99% vs 100% (5,671)"


## Which prepared columns each model leans on

The ten prepared columns each model relies on most: for logistic regression the size of the coefficient (comparable, because the inputs are scaled or 0/1), for the tree models their own importance measure. These describe **what the models learnt from train**. They do **not** show which columns are safe or useful, and they are not used to select inputs (notebook 07, P4).

Watch the provisional inputs and agent `236` (flagged in notebook 10, Step 3). If the models lean heavily on them, the open timing questions matter more.

In [8]:
feature_names = [n.split("__", 1)[1] for n in X_train.columns]
importance = {
    "logistic regression": np.abs(models["logistic regression"].coef_[0]),
    "random forest": models["random forest"].feature_importances_,
    "XGBoost": models["XGBoost"].feature_importances_,
    "CatBoost": models["CatBoost"].get_feature_importance(),
}
top = {name: [f"{feature_names[i]} ({values[i] / values.sum():.0%})" for i in np.argsort(values)[::-1][:10]]
       for name, values in importance.items()}
display(pd.DataFrame(top, index=range(1, 11)))

,logistic regression,random forest,XGBoost,CatBoost
1,agent_236 (4%),previous_cancellations (44%),previous_cancellations (17%),previous_cancellations (70%)
2,agent_281 (3%),market_segment_Groups (10%),market_segment_Groups (6%),market_segment_Groups (10%)
3,company_202 (3%),agent_1 (6%),agent_1 (5%),previous_bookings_not_canceled (3%)
4,agent_162 (3%),hotel_Resort Hotel (5%),agent_240 (4%),hotel_Resort Hotel (3%)
5,company_223 (2%),distribution_channel_TA/TO (3%),distribution_channel_TA/TO (4%),agent_236 (2%)
6,agent_28 (2%),market_segment_Online TA (2%),agent_28 (3%),market_segment_Offline TA/TO (2%)
7,agent_308 (2%),agent_28 (2%),market_segment_Offline TA/TO (2%),distribution_channel_TA/TO (2%)
8,agent_119 (2%),market_segment_Direct (2%),agent_236 (2%),agent_28 (1%)
9,previous_cancellations (2%),market_segment_Offline TA/TO (2%),company_67 (2%),is_repeated_guest (1%)
10,agent_35 (2%),agent_236 (2%),agent_208 (2%),company_none (1%)


## Save the fitted models and the train probabilities

Writes to `data/models/` (inside `data/`, which `.gitignore` keeps out of the repository). These are **training-set fits for the val comparison**, not a chosen model. The model the client receives goes in `models/` only after it is chosen on val and frozen.

| File | Contents |
|---|---|
| `<model>.joblib` | Each fitted model. Predict val with it after transforming val with `data/preprocessed/preprocessor.joblib`. |
| `train_probabilities.csv` | Every model's train probability, by booking `row` |
| `train_scores.csv`, `train_nightly_scores.csv` | The tables above |

In [9]:
model_dir = repo_root / "data" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

for name, model in models.items():
    joblib.dump(model, model_dir / f"{name.replace(' ', '_')}.joblib")
pd.DataFrame(probabilities, index=X_train.index).to_csv(model_dir / "train_probabilities.csv", index_label="row")
scores.to_csv(model_dir / "train_scores.csv", index_label="model")
nightly_scores.to_csv(model_dir / "train_nightly_scores.csv")

# Reload each model and confirm it gives the same probabilities.
for name in models:
    reloaded = joblib.load(model_dir / f"{name.replace(' ', '_')}.joblib")
    assert np.allclose(reloaded.predict_proba(X)[:, 1], probabilities[name]), f"{name}: reloaded model differs"

for path in sorted(model_dir.iterdir()):
    print(f"Saved data/models/{path.name}")

Saved data/models/CatBoost.joblib
Saved data/models/logistic_regression.joblib
Saved data/models/random_forest.joblib
Saved data/models/train_nightly_scores.csv
Saved data/models/train_probabilities.csv
Saved data/models/train_scores.csv
Saved data/models/XGBoost.joblib


## Summary

- **Four models and the flat-rule reference** were fitted on train and measured on train: logistic regression, random forest, XGBoost and CatBoost, all on the same leakage-checked inputs from notebook 10, with fixed settings and no rebalancing.
- **The tables above are in-sample.** They show how well each model fits the bookings it learnt from. They do not show which model will do best on nights it has not seen, and nothing here is a performance or savings claim (project rule 8).
- **No euro cost yet.** That needs an oversell buffer chosen on val under the walk ceiling.
- **Four inputs remain provisional**, so every result inherits their unverified timing.

**Next (val notebook):** transform val with the saved preprocessor, predict with each saved model, and compare the probability scores and the expected-vs-actual nightly error on val. Then choose each candidate's buffer on val under the walk ceiling (10 walks per 1,000 arrivals), and compare euros, empty rooms and walked guests against the flat rule with its own buffer chosen the same way (project rules 5 and 6).